In [1]:
import os
os.chdir('/Users/madsbirch/Documents/4_semester/BAL/bayesian-active-learning')
print("Current working directory: {0}".format(os.getcwd()))

# OPTIONAL: Load the "autoreload" extension so that code can change
%load_ext autoreload
%autoreload 2

Current working directory: /Users/madsbirch/Documents/4_semester/BAL/bayesian-active-learning


In [2]:
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_moons, make_circles, make_classification
from matplotlib.colors import ListedColormap
from mpl_toolkits.axes_grid1 import make_axes_locatable
from sklearn.model_selection import train_test_split

import pickle

import torch
import torchvision
import torchvision.transforms as transforms

import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, SubsetRandomSampler


from src.models.model import MLP, CNN
from src.data.data import get_dataloaders, TwoMoons
from src.models.train_model import train, test
from src.features.utils import random_query, query_the_oracle, plot_decision_bound, softmax_grid, entropy_grid, BALD_grid, var_grid, BALD_1_grid, BALD_2_grid, BALD_grid_viz, BALD_query
from src.visualization.visualize import plot_acquisition_output

torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

# Set device
device = "mps" if torch.backends.mps.is_available() else "cpu"
device = 'cpu'
print(f"Using device: {device}")

FIGURE_PATH = '/Users/madsbirch/Documents/4_semester/BAL/bayesian-active-learning/reports/figures/'

Using device: cpu


In [3]:
# generate data
X, y = make_moons(n_samples = 1000, noise = 0.2, random_state=9)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=9)

traindata = TwoMoons(X_train, y_train, return_idx = True)
testdata = TwoMoons(X_test, y_test, return_idx = True)

# dataloader and dataset class
train_loader, test_loader = get_dataloaders(traindata, testdata, batch_size=256)

# model, optimizer nad hyper parameters'
drop_out = 0.1

model = MLP(drop_out=drop_out)
num_epochs = 200
batch_size = 256

# acrtive learning params
num_queries = 20
query_size = 10
lr = 6e-4

# posterior approx params
T = 1000

# list of sampling strategies
strat_list = ['random', 'margin', 'entropy', 'bald']

init_pool_size = 10
init_pool_idx = np.random.randint(0,500, size = init_pool_size).tolist()

### Ensembles

for e in ensemble:
    train